In [ ]:
# Make this notebook work from fine-tuning/ or fine-tuning/clinical-rtor/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('clinical-rtor', 'pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 03 (Clinical) · Batch abstraction + tool-calling fine-tuning

Run the abstractor row-by-row over surgical cases, force a strict `{is_return_to_or, evidence}` JSON shape, and robustly parse it. The lab compares tool-schema configurations and computes the actual prompt-token change from API usage; no savings percentage is assumed.

---
## Step 1 — Config, client, prompt & schema

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ.get('AZURE_SUBSCRIPTION_ID')
RESOURCE_GROUP           = os.environ.get('AZURE_RESOURCE_GROUP')
RESOURCE_NAME            = os.environ.get('AZURE_RESOURCE_NAME')
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)
print('client ready ->', AZURE_OPENAI_ENDPOINT)


In [ ]:
import json
from pathlib import Path

CASES = [json.loads(l) for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything. If the index OR current operative note documents that
     the second procedure was planned, staged, anticipated, or scheduled at the time of the index
     surgery, then is_return_to_or = false (even if it occurs within 30 days).
  2. Unplanned + related + within 30 days = RTOR. If the current surgery is unplanned and treats a
     complication of the index surgery (bleeding, hematoma, surgical-site infection, wound dehiscence,
     anastomotic leak, abscess, graft/flap failure) within 30 days, then is_return_to_or = true.
  3. Unrelated anatomy or new diagnosis = not RTOR (false), regardless of timing.
  4. Outside the 30-day window = not RTOR (false).

Rule 2 - Operating-room requirement. The return must be to an operating room. Bedside, ICU, IR,
  endoscopy-suite, or clinic procedures do NOT count: is_return_to_or = false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence from the source documents
  verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Evaluate the context against the Specific Abstraction Rules, resolving any conflicting data '
    'using the exact order specified in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- Do not include conversational filler. Do not include markdown formatting like a json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json        = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json           = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note        = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note      = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Safely extract JSON from the LLM response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print(f'Loaded {len(CASES)} labeled cases. Prompt + parser ready.')
print('--- USER PROMPT for', CASES[0]['case_id'], '(first 500 chars) ---')
print(build_user_prompt(CASES[0])[:500])


In [ ]:
import json
from pathlib import Path
RTOR_TOOLS = json.loads(Path('data/rtor_tools_schema.json').read_text(encoding='utf-8'))
print('Loaded', len(RTOR_TOOLS), 'tool schemas:')
for t in RTOR_TOOLS:
    print('  -', t['function']['name'])


---
## Step 2 — Load the cases into a DataFrame

One row per surgical episode — the same shape a registry export or EHR query would hand you.

In [ ]:
import pandas as pd
df = pd.DataFrame(CASES)
print(df.shape)
display(df[['case_id', 'provider_npi', 'index_surgery_procedure_desc', 'current_surgery_procedure_desc', 'is_return_to_or']])


---
## Step 3 — The AI abstraction function (row-wise, strict JSON)

`response_format=json_object` forces valid JSON; `safe_parse` still guards against stray markdown fences. We expand the parsed result into columns and score against the gold label.

In [ ]:
import pandas as pd

def classify(case):
    r = client.chat.completions.create(
        model=BASE_DEPLOYMENT,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',   'content': build_user_prompt(case)}],
        temperature=0.0, max_tokens=300, response_format={'type': 'json_object'},
    )
    return r.choices[0].message.content

# Execute the AI function across the DataFrame (mirrors the production batch job)
df['raw_ai_response'] = df.apply(lambda row: classify(row.to_dict()), axis=1)

# Robustly parse + expand into structured columns
parsed = df['raw_ai_response'].apply(safe_parse).apply(pd.Series)
df['pred_is_return_to_or'] = parsed['is_return_to_or']
df['pred_evidence']        = parsed['evidence']
df['correct']              = df['pred_is_return_to_or'] == df['is_return_to_or']

acc = df['correct'].mean()
print(f'Batch accuracy vs gold: {acc:.0%}  ({int(df["correct"].sum())}/{len(df)})')
display(df[['case_id', 'provider_npi', 'is_return_to_or', 'pred_is_return_to_or', 'correct', 'pred_evidence']])


---
## Step 4 — The token bill you can fine-tune away

If you let the model emit a `classify_return_to_or` **tool call** for downstream systems, you ship the tool schema on every request. Tool-calling fine-tuning bakes that schema into the weights so you can drop it at inference.

In [ ]:
full_tools_json = json.dumps(RTOR_TOOLS)
approx_full = len(full_tools_json) // 4   # ~4 chars per token
print(f'Tool schemas: {len(RTOR_TOOLS)} functions, ~{approx_full} tokens shipped PER call if sent every time.')
print(f'After tool-calling fine-tuning you can drop the tools array entirely: ~{approx_full} fewer tokens/turn.')
print('Across thousands of charts a night, that is the dominant cost lever.')


---
## Step 5 — Write the tool-calling SFT artifact

Baking the schema in is the **same** SFT job as Lab 01 with one change: each record also carries the `tools` array. We emit that artifact here (no billable job submitted); submit it exactly like Lab 01 Step 3 when you want the tuned tool-caller.

In [ ]:
import json
from pathlib import Path

src_path = Path('data/rtor_training.jsonl')
assert src_path.exists(), 'Run Lab 00 first to generate data/rtor_training.jsonl'
src = [json.loads(l) for l in src_path.read_text(encoding='utf-8-sig').splitlines() if l.strip()]

out_path = Path('data/rtor_tools.jsonl')
with open(out_path, 'w', encoding='utf-8-sig') as f:
    for rec in src:
        rec2 = dict(rec)
        rec2['tools'] = RTOR_TOOLS
        f.write(json.dumps(rec2) + '\n')

print(f'Wrote {len(src)} tool-calling records -> {out_path}')
print('Submit with the Lab 01 Step 3 cell, swapping TRAIN for data/rtor_tools.jsonl and suffix "acme-rtor-tools".')


---
## Takeaways

- The whole abstraction job is a **row-wise strict-JSON call + a defensive parser** — exactly the production cell, now reproducible and scored.
- Evidence citations make every decision **auditable** (Lab 17 / Responsible AI).
- Tool-calling fine-tuning turns a per-call schema tax into a one-time training cost.
- Next: **Lab 07** turns batch accuracy into a precision/recall scoreboard.